<a href="https://colab.research.google.com/github/Sathvikabanavath/GenAI-Tasks/blob/main/Week2b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import faiss
import tiktoken
from sentence_transformers import SentenceTransformer

In [ ]:
documents = [
    "Artificial intelligence (AI) refers to systems that mimic human intelligence to perform tasks such as learning, reasoning, and problem solving.",
    "Machine learning is a subset of artificial intelligence focused on algorithms that improve through experience and data.",
    "Deep learning uses multi-layered neural networks to model complex patterns in data.",
    "Natural language processing allows computers to understand and generate human language.",
    "Computer vision focuses on enabling machines to interpret visual information from the world.",
    "Reinforcement learning trains agents to make sequences of decisions by rewarding desired behaviors.",
    "Large language models are trained on massive text corpora and can perform a wide range of NLP tasks.",
    "Transformers are neural network architectures that rely on self-attention mechanisms to process sequences."
]

In [ ]:
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, min_tokens=1, max_tokens=100): # Adjusted min_tokens and max_tokens
    tokens = tokenizer.encode(text)
    chunks = []

    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk = tokens[start:end]
        # Only add chunk if it meets min_tokens requirement, ensuring at least one token
        if len(chunk) >= min_tokens:
            chunks.append(tokenizer.decode(chunk))
        start = end

    return chunks

In [ ]:
chunks = []
for doc in documents:
    chunks.extend(chunk_text(doc))

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 8


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
embeddings = embeddings.reshape(len(embeddings), -1)

dimension = embeddings.shape[-1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Vectors stored in FAISS:", index.ntotal)

Vectors stored in FAISS: 8


In [ ]:
def retrieve(query, top_k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    return [chunks[i] for i in indices[0]]

In [ ]:
results = retrieve("What is deep learning?", top_k=3)

for i, text in enumerate(results, 1):
    print(f"{i}. {text}")

1. Deep learning uses multi-layered neural networks to model complex patterns in data.
2. Machine learning is a subset of artificial intelligence focused on algorithms that improve through experience and data.
3. Artificial intelligence (AI) refers to systems that mimic human intelligence to perform tasks such as learning, reasoning, and problem solving.
